# 16 — Spatial Maps & Descriptive Analysis
## RentSignal — Berlin Rental Market 2026

Interactive maps exploring 8,250 Berlin rental listings with unit-level spatial features.

**Maps:**
1. **Rent heatmap** — rent €/m² across Berlin
2. **Coordinate source quality** — listing vs geocoded vs centroid
3. **Food density** — restaurant/café hotspots (top SHAP predictor)
4. **Vegetation (NDVI)** — green neighborhoods
5. **Transit accessibility** — distance to U-Bahn
6. **Bezirk rent comparison** — median rent by district

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import folium
from folium.plugins import HeatMap, MarkerCluster
import branca.colormap as cm

PROJECT_ROOT = Path('..').resolve()
PROC_DIR = PROJECT_ROOT / 'data' / 'processed'

# Load all tables
units = pd.read_parquet(PROC_DIR / 'units.parquet')
listings = pd.read_parquet(PROC_DIR / 'listings.parquet')
spatial = pd.read_parquet(PROC_DIR / 'spatial_unit.parquet')

# Merge for analysis
df = units.merge(listings[['unit_id', 'rent_sqm', 'baseRent']], on='unit_id')
df = df.merge(spatial, on='unit_id', suffixes=('', '_spatial'))
df = df[df['lat'].notna()].copy()

print(f'Merged dataset: {len(df):,} units')
print(f'Rent range: {df["rent_sqm"].min():.1f} – {df["rent_sqm"].max():.1f} €/m²')
print(f'Mean rent: {df["rent_sqm"].mean():.1f} €/m², Median: {df["rent_sqm"].median():.1f} €/m²')

BERLIN_CENTER = [52.52, 13.405]

## Map 1 — Rent Heatmap (€/m²)

Warmer colors = higher rent. Shows the classic Berlin rent gradient: expensive center (Mitte, Charlottenburg) fading to affordable periphery.

In [ ]:
# Rent heatmap — weighted by rent_sqm
m1 = folium.Map(location=BERLIN_CENTER, zoom_start=11, tiles='CartoDB positron')

# Use only address-precise units for cleaner heatmap
precise = df[df['coord_source'].isin(['listing', 'kaggle_match', 'title_mining'])]
heat_data = precise[['lat', 'lon', 'rent_sqm']].dropna().values.tolist()

HeatMap(
    heat_data,
    min_opacity=0.4,
    radius=12,
    blur=15,
    gradient={0.2: '#2166ac', 0.4: '#67a9cf', 0.6: '#fddbc7', 0.8: '#ef8a62', 1.0: '#b2182b'},
).add_to(m1)

folium.map.LayerControl().add_to(m1)
m1

## Map 2 — Rent Scatter (Color-Coded €/m²)

Each dot is a unit. Color: blue (cheap) → red (expensive). Size: proportional to living space. Click for details.

In [ ]:
# Rent scatter — color by rent_sqm, only precise coords
m2 = folium.Map(location=BERLIN_CENTER, zoom_start=11, tiles='CartoDB positron')

colormap = cm.LinearColormap(
    colors=['#2166ac', '#67a9cf', '#f7f7f7', '#ef8a62', '#b2182b'],
    vmin=5, vmax=30,
    caption='Rent €/m²'
)
colormap.add_to(m2)

# Sample to avoid overwhelming the map (max 3000 points)
sample = precise.sample(min(3000, len(precise)), random_state=42)

for _, row in sample.iterrows():
    color = colormap(min(max(row['rent_sqm'], 5), 30))
    popup_text = (
        f"<b>{row['bezirk']}</b> — {row['plz']}<br>"
        f"€{row['rent_sqm']:.1f}/m² · {row['livingSpace']:.0f}m² · {row['noRooms']:.0f}R<br>"
        f"€{row['baseRent']:.0f}/month<br>"
        f"{row.get('street', '') or ''} {row.get('house_number', '') or ''}"
    )
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=3,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        popup=folium.Popup(popup_text, max_width=250),
    ).add_to(m2)

m2

## Map 3 — Coordinate Source Quality

Shows which units have precise coordinates (blue) vs. area approximations (orange/red). Centroid units cluster at grid points — expected behavior.

In [ ]:
# Coordinate source quality map
m3 = folium.Map(location=BERLIN_CENTER, zoom_start=11, tiles='CartoDB positron')

source_styles = {
    'listing':                {'color': '#2166ac', 'radius': 2, 'opacity': 0.4},
    'kaggle_match':           {'color': '#4daf4a', 'radius': 3, 'opacity': 0.6},
    'title_mining':           {'color': '#ff7f00', 'radius': 4, 'opacity': 0.8},
    'plz_ortsteil_centroid':  {'color': '#e41a1c', 'radius': 3, 'opacity': 0.5},
    'plz_centroid':           {'color': '#984ea3', 'radius': 4, 'opacity': 0.7},
}

# Add each source as a feature group (togglable)
for source, style in source_styles.items():
    fg = folium.FeatureGroup(name=f'{source} ({(df["coord_source"] == source).sum():,})')
    mask = df['coord_source'] == source
    sub = df[mask].sample(min(1500, mask.sum()), random_state=42)
    for _, row in sub.iterrows():
        folium.CircleMarker(
            location=[row['lat'], row['lon']],
            radius=style['radius'],
            color=style['color'],
            fill=True,
            fill_color=style['color'],
            fill_opacity=style['opacity'],
        ).add_to(fg)
    fg.add_to(m3)

folium.LayerControl().add_to(m3)
m3

## Map 4 — Food Density (count_food_1000m)

The #1 SHAP predictor in the v3 model. Colors units by number of restaurants/cafés within 1km. The urban core lights up.

In [ ]:
# Food density map
m4 = folium.Map(location=BERLIN_CENTER, zoom_start=11, tiles='CartoDB dark_matter')

food_cm = cm.LinearColormap(
    colors=['#ffffcc', '#fd8d3c', '#e31a1c', '#800026'],
    vmin=0, vmax=400,
    caption='Restaurants & Cafés within 1km'
)
food_cm.add_to(m4)

sample = precise.sample(min(3000, len(precise)), random_state=42)
for _, row in sample.iterrows():
    val = row.get('count_food_1000m', 0)
    color = food_cm(min(val, 400))
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=3,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        popup=f"Food 1km: {val:.0f}<br>Rent: €{row['rent_sqm']:.1f}/m²",
    ).add_to(m4)

m4

## Map 5 — Vegetation (NDVI 500m buffer)

Green = high vegetation, brown = concrete/built-up. Sentinel-2 satellite data at unit level. Tiergarten, Grunewald, and outer suburbs should glow green.

In [ ]:
# NDVI vegetation map
m5 = folium.Map(location=BERLIN_CENTER, zoom_start=11, tiles='CartoDB positron')

ndvi_cm = cm.LinearColormap(
    colors=['#8c510a', '#d8b365', '#f6e8c3', '#c7eae5', '#5ab4ac', '#01665e'],
    vmin=0.05, vmax=0.50,
    caption='NDVI (500m buffer) — Vegetation Index'
)
ndvi_cm.add_to(m5)

sample = precise[precise['ndvi_500m'].notna()].sample(min(3000, len(precise)), random_state=42)
for _, row in sample.iterrows():
    val = row['ndvi_500m']
    color = ndvi_cm(min(max(val, 0.05), 0.50))
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=3,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        popup=f"NDVI 500m: {val:.3f}<br>Rent: €{row['rent_sqm']:.1f}/m²<br>{row['bezirk']}",
    ).add_to(m5)

m5

## Map 6 — Distance to U-Bahn

Blue = close to U-Bahn, red = far. The U-Bahn network shape should be visible in the data.

In [ ]:
# U-Bahn distance map
m6 = folium.Map(location=BERLIN_CENTER, zoom_start=11, tiles='CartoDB positron')

ubahn_cm = cm.LinearColormap(
    colors=['#08519c', '#3182bd', '#6baed6', '#fee0d2', '#fc9272', '#de2d26'],
    vmin=0, vmax=4000,
    caption='Distance to nearest U-Bahn (m)'
)
ubahn_cm.add_to(m6)

sample = precise.sample(min(3000, len(precise)), random_state=42)
for _, row in sample.iterrows():
    val = row.get('dist_ubahn_m', 0)
    color = ubahn_cm(min(val, 4000))
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=3,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        popup=f"U-Bahn: {val:.0f}m<br>Rent: €{row['rent_sqm']:.1f}/m²",
    ).add_to(m6)

m6

## Descriptive Stats — Spatial Features by Bezirk

In [ ]:
import matplotlib.pyplot as plt

# Bezirk-level summary
bezirk_stats = df.groupby('bezirk').agg(
    n_units=('unit_id', 'count'),
    rent_median=('rent_sqm', 'median'),
    rent_mean=('rent_sqm', 'mean'),
    food_1km_median=('count_food_1000m', 'median'),
    cafe_500m_median=('count_cafe_500m', 'median'),
    ubahn_median=('dist_ubahn_m', 'median'),
    cbd_median=('dist_cbd_m', 'median'),
    ndvi_500m_median=('ndvi_500m', 'median'),
).sort_values('rent_median', ascending=False)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# 1. Median rent by bezirk
bezirk_stats['rent_median'].plot(kind='barh', ax=axes[0,0], color='steelblue')
axes[0,0].set_title('Median Rent €/m² by Bezirk')
axes[0,0].set_xlabel('€/m²')

# 2. Food density by bezirk
bezirk_stats.sort_values('food_1km_median', ascending=False)['food_1km_median'].plot(
    kind='barh', ax=axes[0,1], color='#e31a1c')
axes[0,1].set_title('Median Food Venues (1km) by Bezirk')

# 3. Distance to U-Bahn by bezirk
bezirk_stats.sort_values('ubahn_median')['ubahn_median'].plot(
    kind='barh', ax=axes[0,2], color='#08519c')
axes[0,2].set_title('Median Distance to U-Bahn (m)')

# 4. NDVI by bezirk
bezirk_stats.sort_values('ndvi_500m_median', ascending=False)['ndvi_500m_median'].plot(
    kind='barh', ax=axes[1,0], color='#01665e')
axes[1,0].set_title('Median NDVI (500m) by Bezirk')

# 5. Rent vs Food scatter
ax = axes[1,1]
ax.scatter(bezirk_stats['food_1km_median'], bezirk_stats['rent_median'], s=bezirk_stats['n_units']/5, alpha=0.7)
for bzk, row in bezirk_stats.iterrows():
    ax.annotate(bzk, (row['food_1km_median'], row['rent_median']), fontsize=7, ha='center')
ax.set_xlabel('Food Venues (1km)')
ax.set_ylabel('Median Rent €/m²')
ax.set_title('Rent vs Food Density by Bezirk')

# 6. Rent vs CBD distance
ax = axes[1,2]
ax.scatter(bezirk_stats['cbd_median']/1000, bezirk_stats['rent_median'], s=bezirk_stats['n_units']/5, alpha=0.7)
for bzk, row in bezirk_stats.iterrows():
    ax.annotate(bzk, (row['cbd_median']/1000, row['rent_median']), fontsize=7, ha='center')
ax.set_xlabel('Distance to CBD (km)')
ax.set_ylabel('Median Rent €/m²')
ax.set_title('Rent vs CBD Distance by Bezirk')

plt.suptitle('Berlin Rental Market 2026 — Spatial Feature Analysis', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# Print the table
print(bezirk_stats.round(1).to_string())

## Correlation Matrix — Spatial Features vs Rent

In [ ]:
# Correlation of spatial features with rent
spatial_features = [
    'dist_cbd_m', 'dist_transit_m', 'dist_ubahn_m', 'dist_park_m', 'dist_water_m', 'dist_school_m',
    'count_food_500m', 'count_food_1000m', 'count_restaurant_1000m', 'count_cafe_500m',
    'count_shop_1000m', 'count_transit_1000m', 'count_building_200m',
    'ndvi_100m', 'ndvi_500m', 'ndwi_500m', 'ndbi_500m',
]

# Only use precise coords for unbiased correlations
precise_df = df[df['coord_source'].isin(['listing', 'kaggle_match', 'title_mining'])]
corrs = precise_df[spatial_features + ['rent_sqm']].corr()['rent_sqm'].drop('rent_sqm').sort_values()

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#b2182b' if v < 0 else '#2166ac' for v in corrs.values]
corrs.plot(kind='barh', ax=ax, color=colors)
ax.set_title('Correlation with Rent €/m² (precise coordinates only)', fontsize=13)
ax.set_xlabel('Pearson r')
ax.axvline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

print('Top positive correlations:')
for feat, r in corrs.tail(5).items():
    print(f'  {feat:<28} r = {r:+.3f}')
print('\nTop negative correlations:')
for feat, r in corrs.head(5).items():
    print(f'  {feat:<28} r = {r:+.3f}')